# Train real matched controls and load reference baselines

Formal DiffJev workflow; see AGENTS.md and prompt.md.

In [ ]:
import os, sys, gc
from pathlib import Path
ROOT = Path(os.environ.get('DIFFJEV_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'AGENTS.md').exists(), 'Set DIFFJEV_ROOT to the checked-out repository'
sys.path.insert(0, str(ROOT / 'src'))
from diffjev import workflows as w
from diffjev.config import load_yaml_config
cfg = load_yaml_config(str(ROOT / 'configs/workflow.yaml'))
SEED, DEVICE = cfg['seed'], cfg['device']


Every trained control starts again from the original pinned checkpoint. The released Jeva model uses its official runtime; the AR reference uses Qwen2.5-3B-Instruct. Missing assets cause an explicit error. API calls remain disabled unless enable_paid_api is deliberately enabled in the shared configuration.

In [ ]:
for backbone in cfg['backbones']:
    methods = ['serialized_dlm'] + (['ar'] if backbone.startswith('nemotron') else [])
    for method in methods:
        print(w.train_run(backbone, method=method, seed=SEED, device=DEVICE))
        gc.collect()

In [ ]:
for name in cfg['references'] + ['bge']:
    runner = w.baseline_runner(name, device=DEVICE)
    print(name, type(runner).__name__)
    del runner
    gc.collect()